# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [4]:
# Start with some imports - rich is a library for making formatted text output in the terminal

import os
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [1]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [5]:
GPT_4_MINI_MODEL = "openai/gpt-4o-mini"
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_url = "https://openrouter.ai/api/v1"
openai = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)

In [6]:
# Some lists!

todos = []
completed = []

In [7]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [8]:
get_todo_report()

''

In [9]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [10]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [11]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [12]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [13]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [14]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [15]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [16]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [17]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model=GPT_4_MINI_MODEL, messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [18]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [19]:
todos, completed = [], []
loop(messages)

Todo #1: Determine the time difference between the trains' departures.
Todo #2: Calculate the distance traveled by the Boston train before the New York train departs.
Todo #3: Set up an equation to represent the distances of both trains when they meet.
Todo #4: Solve the equation to find the time when they meet.

The time difference between the departures is 1 hour; the Boston train leaves at 2:00 pm and the New York train 
leaves at 3:00 pm.

Todo #1: Determine the time difference between the trains' departures.
Todo #2: Calculate the distance traveled by the Boston train before the New York train departs.
Todo #3: Set up an equation to represent the distances of both trains when they meet.
Todo #4: Solve the equation to find the time when they meet.

The Boston train travels for 1 hour before the New York train departs, traveling 60 miles in that hour (60 mph * 1 
hour).

Todo #1: Determine the time difference between the trains' departures.
Todo #2: Calculate the distance traveled by the Boston train before the New York train departs.
Todo #3: Set up an equation to represent the distances of both trains when they meet.
Todo #4: Solve the equation to find the time when they meet.

Let t be the time in hours after 3:00 pm when they meet. The distance from Boston is given by: distance = speed * 
time. The Boston train will travel for t + 1 hours and the New York train will travel for t hours. So, the equation
is: 60(t + 1) = 80t.

Todo #1: Determine the time difference between the trains' departures.
Todo #2: Calculate the distance traveled by the Boston train before the New York train departs.
Todo #3: Set up an equation to represent the distances of both trains when they meet.
Todo #4: Solve the equation to find the time when they meet.

Solving the equation, we get: 60t + 60 = 80t -> 20t = 60 -> t = 3. The trains meet 3 hours after 3:00 pm, which is 
6:00 pm.

Todo #1: Determine the time difference between the trains' departures.
Todo #2: Calculate the distance traveled by the Boston train before the New York train departs.
Todo #3: Set up an equation to represent the distances of both trains when they meet.
Todo #4: Solve the equation to find the time when they meet.

The two trains will meet at **6:00 pm**.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>